In [ ]:
# 1. Clone your repository directly into the Kaggle working directory
!git clone https://github.com/peri-7/cv_project_saliency.git

import sys
import os
import torch

# 2. Append the cloned repository folder to Python's path
sys.path.append('/kaggle/working/cv_project_saliency/')

# 3. Verify hardware acceleration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Cloud Hardware active: {device}")

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader
import torch.optim.lr_scheduler as lr_scheduler
import torchvision.transforms as transforms
from tqdm import tqdm

from src.dataset import RawDataset, FeatureDataset
from src.models import ResNet
from src.decoder import Decoder
from src.losses import Composite_Loss
from src.training import train_one_epoch, evaluate_model, test_model

In [ ]:
print("---  Phase 1: Feature Extraction ---")

# Standardize high-resolution input for the ResNet backbone
image_transform = transforms.Compose([
    transforms.Resize((480, 640)), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

extractor = ResNet().to(device)

# We will loop through both the Train and Val splits
# (Assuming the Kaggle dataset is structured as /kaggle/input/salicon/images/train, etc.)
base_input_path = '/kaggle/input/datasets/hughiephan/salicon-mini'
splits = ['train', 'val']

with torch.no_grad():
    for split in splits:
        os.makedirs(f"/kaggle/working/features/{split}", exist_ok=True)
        
        raw_dataset = RawDataset(
            image_dir=os.path.join(base_input_path, f"images/{split}"), 
            transform=image_transform
        )
        
        raw_loader = DataLoader(raw_dataset, batch_size=16, shuffle=False, num_workers=2)
        
        print(f"Extracting {split} set...")
        for images, filenames in tqdm(raw_loader, desc=f"{split} Extraction"):
            images = images.to(device)
            features_dict = extractor(images)
            
            for i in range(images.size(0)):
                base_name = os.path.splitext(filenames[i])[0]
                save_path = f"/kaggle/working/features/{split}/{base_name}.pt"
                
                # Using your optimized indexing trick: v[i] instead of v[i:i+1]
                single_feature_dict = {
                    k: v[i].cpu().clone() for k, v in features_dict.items()
                }
                torch.save(single_feature_dict, save_path)

print("Phase 1 Complete. All tensors cached safely to SSD.")

In [ ]:
print("---  Phase 2: Decoder Training ---")

# Ground truth standardization
map_transform = transforms.Compose([
    transforms.Resize((480, 640)),
    transforms.ToTensor()
])

# Initialize the Dataloaders pointing to your new cached features
train_dataset = FeatureDataset(
    features_dir="/kaggle/working/features/train",
    maps_dir=os.path.join(base_input_path, "maps/train"),
    fixations_dir=os.path.join(base_input_path, "fixations/train"),
    map_transform=map_transform
)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)

val_dataset = FeatureDataset(
    features_dir="/kaggle/working/features/val",
    maps_dir=os.path.join(base_input_path, "maps/val"),
    fixations_dir=os.path.join(base_input_path, "fixations/val"),
    map_transform=map_transform
)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

# Architecture Initialization
decoder = Decoder(in_channels_list=extractor.out_channels, hidden_dim=128).to(device)
criterion = Composite_Loss().to(device)
#optimizer = optim.AdamW(decoder.parameters(), lr=1e-4, weight_decay=1e-4)
optimizer = optim.Adam(decoder.parameters(), lr=1e-4)
epochs = 10
scheduler = lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.01, total_iters=epochs)

# Training Loop with Early Stopping
patience = 0
val_min = float('inf')

for epoch in range(epochs):   
    
    current_lr = scheduler.get_last_lr()[0]
    
    train_loss, kld, cc = train_one_epoch(decoder, train_loader, optimizer, criterion, device)
    val_loss = evaluate_model(decoder, val_loader, criterion, device)
    
    print(f"Epoch {epoch} | LR: {current_lr:.3e} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    scheduler.step()
    
    # Save the absolute best weights to the hard drive
    if val_loss < val_min:
        patience = 0
        val_min = val_loss
        torch.save(decoder.state_dict(), "/kaggle/working/best_resnet_decoder.pth")
        print("  -> New best model saved!")
    else:
        patience += 1
        
    if patience > 3: 
        print(f"Early Stopping triggered on Epoch {epoch}. Restoring best weights.")
        decoder.load_state_dict(torch.load("/kaggle/working/best_resnet_decoder.pth"))
        break   

print("-" * 50)
print("Training complete.")

In [ ]:
print("--- Phase 3: Final Evaluation ---")

# Run the strict metric calculation on the validation proxy set
avg_loss, avg_kld, avg_cc, avg_nss, avg_auc, avg_ig = test_model(decoder, val_loader, criterion, device)

print(f"Final Model Benchmark (ResNet-50 Backbone):")
print(f"Composite Loss: {avg_loss:.4f}")
print(f"KLD: {avg_kld:.4f}")
print(f"CC:  {avg_cc:.4f}")
print(f"NSS: {avg_nss:.4f}")
print(f"AUC: {avg_auc:.4f}")
print(f"IG:  {avg_ig:.4f} bits")